## Interpretation: `main_simulate.py`

### Notebook Disclaimer

**This notebook is for interpretation and documentation purposes only. It should not be executed.**

The code blocks shown in this notebook are excerpts from [`main_simulate.py`](main_simulate.py), which is a module imported and called by [`TombeZhu2019.ipynb`](TombeZhu2019.ipynb). This notebook serves to explain, block by block, the logic of the equilibrium function `eqm()` defined in `main_simulate.py`.

### 1. Load the Parameters

In [ ]:
# basic parameter
N       = params["N"]           # 30 provinces + 1 row

# labor distribution (from data)
La      = params["La"]          # agricultural labor distribution after migration
Ln      = params["Ln"]          # non-agricultural labor distribution after migration
L1base  = params["L1base"]      # same as L1, labor distribution after migration
L0      = params["L0"]          # hukou-registered labor distribution, backed out by L1 and mij

# income & revenue (from Gauss-Seidel iteration)
Ir      = params["Ir"]          # income in agricultural sector
Iu      = params["Iu"]          # income in non-agricultural sector
R_ag    = params["R_ag"]        # revenue of agricultural sector 
R_na    = params["R_na"]        # revenue of non-agricultural sector

# production parameters
theta   = params["theta"]       # the inverse dispersion of productivity
beta_ag = params["beta_ag"]     # share of labor in agricultural production
beta_na = params["beta_na"]     # share of labor in non-agricultural production
eta_ag  = params["eta_ag"]      # share of land in agricultural production
eta_na  = params["eta_na"]      # share of land in non-agricultural production
sigma   = params["sigma"]       # matrix: share of intermediate materials (defined differently from the article)

# preference parameters
kappa   = params["kappa"]       # the inverse dispersion of location preference
alpha   = params["alpha"]       # goods share in utility
psi     = params["psi"]         # agricultural goods share in the goods demand

# trade share
pi_ag   = params["pi_ag"]       # trade shares (agriculture)
pi_na   = params["pi_na"]       # trade shares (non-agriculture)

# migration-related
Vi      = params["Vi"]          # real income per worker
mij2000 = params["mij2000"]     # migration share
Cnijk   = params["Cnijk"]       # migration cost (mu)

# changes
dTa     = params["dTa"]         # hat T_ag      1 in default
dTn     = params["dTn"]         # hat T_na      1 in default
dni_ag  = params["dni_ag"]      # hat tau^ag    1 in default
dni_na  = params["dni_na"]      # hat tau^na    1 in default

### 2. Endogenous Variables

In [ ]:
X = X.reshape(-1, 1)    # 6N endogenous variables in total

dwa = X[0:N]            # hat w_ag
dwn = X[N:2 * N]        # hat w_na
dPa = X[2 * N:3 * N]    # hat P_ag
dPn = X[3 * N:4 * N]    # hat P_na
dLa = X[4 * N:5 * N]    # hat L_ag
dLn = X[5 * N:6 * N]    # hat L_na
dL  = np.column_stack([dLa, dLn]).reshape(-1, 1)

### 3. Equilibrium

#### 3.1 Equilibrium Definition

- The equilibrium is defined differently here.

- The equilibrium defined here is a vector of wages $\{w_n^j\}$, an allocation of labor $\{L_n^j\}$, and a vector of price indices $\{P_n^j\}$ such that

    - Goods market clears.
    $$X^{ag'} = D_r^{ag'} + D_u^{ag'} + (1-\beta^{ag}-\eta^{ag})\sigma^{ag,ag} R^{ag'} + (1-\beta^{na}-\eta^{na})\sigma^{na,ag} R^{na'}$$
    $$X^{na'} = D_r^{na'} + D_u^{na'} + (1-\beta^{na}-\eta^{na})\sigma^{na,na} R^{na'} + (1-\beta^{ag}-\eta^{ag})\sigma^{ag,na} R^{ag'}$$

    - The price indices are calculated by 
    $$P_n^j = \gamma \left[ \sum_{i=1}^{N+1} T_i^j \left(\tau_{ni}^j c_i^j\right)^{-\theta} \right]^{-1/\theta}.$$

    - The labor market clears.

- Finally, these conditions form into $6N + 5$ independent equations, with $6N + 5$ endogenous variables to be solved (normalizing one of the wages).

#### 3.2 Equilibrium-Relevant Calculations

- Goods market clearing
    - To calculate $X_i^{j'}$, we can use the following equation:
        $$R_n^{j'} = \sum_{i=1}^{N+1}\pi_{in}^{j'}X_i^{j'}.$$
        - New revenue $R_n^{j'}$ and new trade shares $\pi_{in}^{j'}$ (or $\hat{\pi}_{in}^j$) need to be calculated.
    - $D_r^{ag'}$, $D_u^{ag'}$, $D_r^{na'}$, $D_u^{na'}$: 
    \begin{align*}
        & D_{r}^{ag'} = \alpha \psi I_r', \\
        & D_{u}^{ag'} = \alpha \psi I_u', \\
        & D_{r}^{na'} = \alpha (1-\psi) I_r', \\
        & D_{u}^{na'} = \alpha (1-\psi) I_u'.
    \end{align*}
        - New income $I_r'$ and $I_u'$ need to be calculated.
- Price indices
    - We'll see that the calculation of both $\hat{P}_n^j$ and $\hat{\pi}_{in}^j$ require $\hat{c}_n^j$.
    \begin{equation}
		\hat{c}_n^j = (\hat{w}_n^j)^{\beta^j}(\hat{r}_n^j)^{\eta^j}(\hat{P}_n^{ag})^{\sigma^{j,ag}}(\hat{P}_n^{na})^{\sigma^{j,na}}.
		\nonumber
	\end{equation}
        - The change of land rent $\hat{r}_n^j$ needs to be calculated.
- Labor market clearing
    - To calculate the new migration share $m_{ni}^{jk'}$, we need the new real income $V'$.

### 4. Calculations

In [ ]:
# changes in land rent
dra = dwa * dLa 
drn = dwn * dLn

The total land income is
$$r_n^j\bar{S}_n^j = [\frac{(1-\alpha)\beta^j + \eta^j}{\alpha\beta^j}]w_n^jL_n^j.$$
Therefore, the changes in land rent can be calculated using 
$$\hat{r}_n^j = \hat{w}_n^j\hat{L}_n^j.$$

In [ ]:
# new revenue
R_ag_new = dwa * dLa * R_ag
R_na_new = dwn * dLn * R_na

According the Cobb-Douglas production function, we have 
$$\beta^jR_n^j = w_n^jL_n^j.$$
Therefore, the new revenue can be calculated using 
$$R_n^{j'} = \frac{w_n^{j'}}{w_n^j}\frac{L_n^{j'}}{L_n^j}R_n^j = \hat{w}_n^j\hat{L}_n^jR_n^j.$$

In [ ]:
# new income 
Ir_new = (beta_ag + eta_ag) * R_ag_new / alpha
dIr = Ir_new / Ir
Iu_new = (beta_na + eta_na) * R_na_new / alpha
dIu = Iu_new / Iu
# calculated using the same equation as in "TombeZhu2019.ipynb", but with new revenue

# new demand
Da_new_r = alpha * psi * Ir_new
Da_new_u = alpha * psi * Iu_new
Dn_new_r = (alpha * (1 - psi)) * Ir_new
Dn_new_u = (alpha * (1 - psi)) * Iu_new

In [ ]:
# new trade shares
inside_ag = (dwa.T ** beta_ag
             * (dPa.T ** sigma[0, 0] * dPn.T ** sigma[0, 1]) ** (1 - beta_ag - eta_ag)
             * dra.T ** eta_ag
             / (dTa.T ** (1.0 / theta)))
kron_inside_ag = np.kron(inside_ag, np.ones((N, 1)))
denominator_ag = np.sum(pi_ag * (dni_ag * kron_inside_ag) ** (-theta), axis=1, keepdims=True)
pi_ag_new = (pi_ag * (dni_ag * kron_inside_ag) ** (-theta)) / np.kron(denominator_ag, np.ones((1, N)))

inside_na = (dwn.T ** beta_na
             * (dPa.T ** sigma[1, 0] * dPn.T ** sigma[1, 1]) ** (1 - beta_na - eta_na)
             * drn.T ** eta_na
             / (dTn.T ** (1.0 / theta)))
kron_inside_na = np.kron(inside_na, np.ones((N, 1)))
denominator_na = np.sum(pi_na * (dni_na * kron_inside_na) ** (-theta), axis=1, keepdims=True)
pi_na_new = (pi_na * (dni_na * kron_inside_na) ** (-theta)) / np.kron(denominator_na, np.ones((1, N)))

Using the same method of calculating the changes in price index, we can calculate the changes in trade share:
$$\hat{\pi}_{ni}^j = \frac{\hat{T}_i^j(\hat{\tau}_{ni}^j\hat{c}_i^j)^{-\theta}}{\sum_{m=1}^{N+1}\pi_{nm}^j\hat{T}_m^j(\hat{\tau}_{nm}^j\hat{c}_m^j)^{-\theta}}.$$
We have to calculate $\hat{c}_n^j$ first:
$$\hat{c}_n^j = (\hat{w}_n^j)^{\beta^j}(\hat{r}_n^j)^{\eta^j}(\hat{P}_n^{ag})^{\sigma^{j,ag}}(\hat{P}_n^{na})^{\sigma^{j,na}}.$$

Code concordance:
$$inside_i^j = (\hat{w}_i^j)^{\beta^j}(\hat{r}_i^j)^{\eta^j}(\hat{P}_i^{ag})^{\sigma^{j,ag}}(\hat{P}_i^{na})^{\sigma^{j,na}} = \hat{c}_i^j / (\hat{T}_i^j)^{\frac{1}{\theta}}.$$
$$denominator_n^j = \sum_{m=1}^{N+1}\pi_{nm}^j(\hat{\tau}_{nm}^j\hat{c}_m^j / (\hat{T}_m^j)^{\frac{1}{\theta}})^{-\theta} = \sum_{m=1}^{N+1}\pi_{nm}^j\hat{T}_m^j(\hat{\tau}_{nm}^j\hat{c}_m^j)^{-\theta}.$$

In [ ]:
# new expenditures
X_ag_new = np.linalg.solve(pi_ag_new.T, R_ag_new)
X_na_new = np.linalg.solve(pi_na_new.T, R_na_new)

In [ ]:
# calculate new migration shares

# first calculate new real income
dP = dPa ** psi * dPn ** (1 - psi)
dVa = dwa / (dP ** alpha * dra ** (1 - alpha))
dVn = dwn / (dP ** alpha * drn ** (1 - alpha))  # calculated according to the consumer price index
dV = np.column_stack([dVa, dVn]).reshape(-1, 1)

# solving for the new migration share using Gauss-Seidel iteration
NN = 2 * (N - 1)
mnn1 = np.diag(mij2000).reshape(-1, 1)
mnn0 = np.ones((NN, 1))
mij_loc = None
while ((mnn0 - mnn1) ** 2).sum() > 1e-20:
    mnn0 = mnn1.copy()
    M_mL_term = (L1base * dL[0:NN]) / (mnn0 * L0)
    temp_local = M_mL_term.reshape(-1, 2)
    M_mL_term_ag_l = temp_local[:, 0]
    M_mL_term_na_l = temp_local[:, 1]
    Cnnjj_ag_l = 1 + (eta_ag + (1 - alpha) * beta_ag) * M_mL_term_ag_l / (alpha * beta_ag)
    Cnnjj_na_l = 1 + (eta_na + (1 - alpha) * beta_na) * M_mL_term_na_l / (alpha * beta_na)
    Cnnjj_l = np.column_stack([Cnnjj_ag_l, Cnnjj_na_l]).reshape(-1, 1)
    cij_l = Cnijk + np.eye(NN) * np.tile(Cnnjj_l, (1, NN))
    Vjmat_l = np.tile((dV[0:NN] * Vi).T, (NN, 1))
    mij_loc = (cij_l * Vjmat_l) ** kappa
    mij_loc = mij_loc / mij_loc.sum(axis=1, keepdims=True)
    mnn1 = np.mean(np.column_stack([np.diag(mij_loc).reshape(-1, 1), mnn0]), axis=1, keepdims=True)

Why do we need to use the Gauss-Seidel iteration?
- Check again the equation for the migration share:
$$m_{ni}^{jk} = \frac{(V_i^k\delta_{ni}^{jk} / \mu_{ni}^{jk})^{\kappa}}{\sum_{k'}\sum_{i'}(V_{i'}^{k'}\delta_{ni'}^{jk'}/ \mu_{ni'}^{jk'})^{\kappa}}.$$
- The opportunity cost of migration brought by land tenure:
\begin{equation}
	\delta_{ni}^{jk} = \left\{
		\begin{aligned}
			&1 + (\frac{(1-\alpha)\beta^j + \eta^j}{\alpha\beta^j})\frac{L_n^j}{L_{nn}^{jj}}, &\text{if}\ n = i\ \text{and}\ j=k; \\
			&1, &\text{if}\ n \neq i\ \text{or}\ j\neq k.
		\end{aligned}
	\right.
	\nonumber
\end{equation}
- The distribution of labor $\Rightarrow$ $\delta_{ni}^{jk}$ $\Rightarrow$ migration $\Rightarrow$ the distribution of labor.

Code Concordance:
- $Cnnjj\_l$ gives the land rebatement adjustment parameter $\delta_{nn}^{jj}$ (since other $\delta$ equals one in other scenarios).
- $Cnijk$ is a matrix with zeros on the diagonal and off-diagonal elements equal to the corresponding $1/\mu$.
- $cij\_l$ is a matrix with diagonal elements equal to the corresponding $\delta$ and off-diagonal elements equal to the corresponding $1/\mu$.

In [ ]:
# new labour allocation
L1_loc = (L0.T @ mij_loc).T

### 5. The Equation System

In [ ]:
F = np.vstack([

    # price index 
    dPa - denominator_ag ** (-1.0 / theta),
    dPn - denominator_na ** (-1.0 / theta),

    # migration share
    L1_loc[0:NN] / L1base[0:NN] - dL[0:NN],

    # no immigration
    np.array([[(dLa[N - 1, 0] * La[N - 1, 0] + dLn[N - 1, 0] * Ln[N - 1, 0]
                - (La[N - 1, 0] + Ln[N - 1, 0]))]]),

    # normalization (Walras' Law)
    np.array([[dwa[N - 1, 0] - dwn[N - 1, 0]]]),

    # trade balance
    X_ag_new - (Da_new_r + Da_new_u
                + (1 - beta_ag - eta_ag) * sigma[0, 0] * R_ag_new
                + (1 - beta_na - eta_na) * sigma[1, 0] * R_na_new),
    X_na_new - (Dn_new_r + Dn_new_u
                + (1 - beta_na - eta_na) * sigma[1, 1] * R_na_new
                + (1 - beta_ag - eta_ag) * sigma[0, 1] * R_ag_new),
])